# 02 — KPIs operativos y scorecards

## Pregunta de negocio
¿Qué rutas y buques muestran peor desempeño cuando las cancelaciones forman parte del denominador?

## Definiciones

- **OTR≤15 de completados:** puntuales / completados.
- **Fiabilidad operacional:** puntuales / programados.
- **Cancel Rate:** cancelados / programados.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.analysis import enrich, executive_kpis, route_scorecard, vessel_rotation_scorecard

In [2]:
df = pd.read_csv(ROOT / 'data/raw/ferry_operations_synthetic.csv', parse_dates=['scheduled_start','actual_start','scheduled_end','actual_end'])
df = enrich(df)
print(df.shape)

(12000, 50)


In [3]:
kpis = executive_kpis(df)
display(kpis.T.rename(columns={0:'valor'}))

,valor
scheduled_services,1.200000e+04
completed_services,1.188600e+04
canceled_services,1.140000e+02
cancel_rate_pct,9.500000e-01
otr_5_completed_pct,6.243000e+01
otr_15_completed_pct,8.906000e+01
operational_reliability_pct,8.822000e+01
avg_departure_delay_min,6.590000e+00
p95_departure_delay_min,2.360000e+01
avg_arrival_delay_min,1.616000e+01


In [4]:
routes = route_scorecard(df)
cols = ['route_id','scheduled_services','completed_services','canceled_services','cancel_rate_pct','otr_15_completed_pct','operational_reliability_pct','avg_delay_min','p95_delay_min','avg_margin_eur','status']
display(routes[cols])

,route_id,scheduled_services,completed_services,canceled_services,cancel_rate_pct,otr_15_completed_pct,operational_reliability_pct,avg_delay_min,p95_delay_min,avg_margin_eur,status
1,BCN-PMI,1504,1486,18,1.20,86.94,85.90,7.46,26.10,23922.68,AMBER
0,BCN-IBZ,1184,1177,7,0.59,86.83,86.32,7.55,27.64,24214.42,AMBER
4,DEN-PMI,1399,1378,21,1.50,88.90,87.56,6.72,25.03,20387.47,AMBER
6,VAL-IBZ,1562,1544,18,1.15,88.73,87.71,7.19,24.68,22381.22,AMBER
5,MAH-BCN,1214,1206,8,0.66,88.72,88.14,6.72,23.32,23832.72,AMBER
7,VAL-PMI,1442,1438,4,0.28,88.46,88.21,6.80,23.61,23975.79,AMBER
2,DEN-FOR,1626,1609,17,1.05,91.17,90.22,5.58,20.46,13330.35,AMBER
3,DEN-IBZ,2069,2048,21,1.01,91.21,90.29,5.45,19.40,14689.58,AMBER


In [5]:
plot_df = routes.sort_values('operational_reliability_pct')
ax = plot_df.plot(x='route_id', y=['otr_15_completed_pct','operational_reliability_pct'], kind='bar', figsize=(11,5), title='OTR frente a fiabilidad total')
ax.set_ylabel('%'); plt.xticks(rotation=35); plt.tight_layout(); plt.show()

In [6]:
vessels = vessel_rotation_scorecard(df)
display(vessels.head(12))
vessels.head(12).sort_values('total_propagated_delay_min').plot(x='vessel_id', y='total_propagated_delay_min', kind='barh', figsize=(9,5), legend=False, title='Buques con mayor retraso propagado')
plt.xlabel('Minutos'); plt.tight_layout(); plt.show()

,vessel_id,vessel_type,scheduled_services,completed_services,cancel_rate_pct,avg_delay_min,p95_delay_min,rotation_impacted_pct,total_propagated_delay_min,avg_scheduled_gap_min,total_margin_eur
2,ECO-03,ECO,444,438,1.35,7.83,25.71,9.68,706.0,827.23,8644049.86
27,ROPAX-08,RO_PAX,415,414,0.24,7.82,28.57,9.40,576.9,894.78,13383637.48
26,ROPAX-07,RO_PAX,428,426,0.47,6.93,25.02,8.88,485.2,863.29,13394992.57
3,ECO-04,ECO,446,443,0.67,7.79,25.09,9.64,465.8,821.53,9124899.84
22,ROPAX-03,RO_PAX,442,440,0.45,6.26,20.61,8.37,460.6,813.44,14306793.67
8,ECO-09,ECO,461,458,0.65,6.94,23.23,8.89,450.5,792.95,9107984.05
1,ECO-02,ECO,467,460,1.50,6.77,23.33,8.99,449.4,769.06,9764096.70
28,ROPAX-09,RO_PAX,403,397,1.49,7.45,25.06,6.70,411.1,932.87,12508434.96
5,ECO-06,ECO,439,433,1.37,6.87,28.54,10.25,400.4,836.99,8756945.95
23,ROPAX-04,RO_PAX,418,414,0.96,6.97,25.40,8.85,388.2,869.82,13288610.76


## Conclusión
La fiabilidad operacional ofrece una lectura más exigente que el OTR de completados porque incorpora cancelaciones.